# ⚽ Football Vision — Analisi Tattica su GPU (Colab)

Codice + modelli + analisi su **GPU gratuita**. In output: **report PDF unico** con
formazione, andamento tattico, heatmap, dashboard e metriche (atletiche/tattiche).

## Come si usa
1. **Runtime → Cambia tipo di runtime → GPU (T4)** → Salva.
2. Esegui le celle in ordine (`Shift+Invio`).
3. Per la clip: **carica un file** (opzione A) — è la più affidabile.
4. Lancia l'analisi e scarica il report.

## 1) Verifica GPU

In [ ]:
import torch
print('✅ GPU attiva:', torch.cuda.get_device_name(0)) if torch.cuda.is_available() else print('❌ GPU NON attiva! Runtime → Cambia tipo di runtime → GPU (T4).')

## 2) Scarica codice + modelli + librerie (~1-2 min)

In [ ]:
!pip -q install ultralytics supervision scikit-learn 2>/dev/null
import os, shutil
if os.path.exists('football-vision'):
    shutil.rmtree('football-vision')
!git clone -q https://github.com/sebavidal2001/football-vision.git
%cd football-vision
import urllib.request
os.makedirs('vista_tattica', exist_ok=True); os.makedirs('clips_input', exist_ok=True)
modelli = {
    'vista_tattica/yolo-football-pitch-detection.pt':
        'https://huggingface.co/martinjolif/yolo-football-pitch-detection/resolve/main/yolo-football-pitch-detection.pt',
    'vista_tattica/giocatori_calcio.pt':
        'https://huggingface.co/uisikdag/yolo-v8-football-players-detection/resolve/main/best.pt',
}
for dst, url in modelli.items():
    if not os.path.exists(dst):
        print('Scarico', os.path.basename(dst), '...'); urllib.request.urlretrieve(url, dst)
print('\n✅ Tutto pronto.')

## 3) Carica la clip (opzione A — consigliata)
Ritaglia prima una clip col programma `AVVIA_ritaglia.bat`, poi caricala qui.
Per una **partita intera** usa l'opzione Google Drive (cella sotto).

In [ ]:
from google.colab import files
import os
os.makedirs('clips_input', exist_ok=True)
up = files.upload()
src = list(up.keys())[0]
!ffmpeg -y -i "{src}" -c:v libx264 -pix_fmt yuv420p -an clips_input/clip_input.mp4 2>/dev/null
print('\n✅ Caricata e normalizzata.')

### Opzione B — Google Drive (per PARTITE INTERE)
Carica il video su Drive, poi:

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os

# Mostra i file presenti in "Il mio Drive" (per trovare il nome esatto del video)
print('📁 File/cartelle in MyDrive:')
for f in sorted(os.listdir('/content/drive/MyDrive')):
    print('   ', f)

# 👉 Scrivi qui il nome ESATTO del tuo video (poi ri-esegui la cella):
PERCORSO = '/content/drive/MyDrive/partita.mp4'   # es. /content/drive/MyDrive/PSG_Inter.mp4
#   se è dentro una cartella:  /content/drive/MyDrive/Calcio/PSG_Inter.mp4

if os.path.exists(PERCORSO):
    os.makedirs('clips_input', exist_ok=True)
    !ffmpeg -y -i "{PERCORSO}" -c:v libx264 -pix_fmt yuv420p -an clips_input/clip_input.mp4 2>/dev/null
    print('\n✅ Video pronto da Drive:', PERCORSO)
else:
    print('\n⚠ File non trovato:', PERCORSO)
    print('   Controlla il nome nella lista qui sopra e correggi PERCORSO.')

## 4) Analisi su GPU
**Clip breve (1-5 min):** `SALTO=2`, `OGNI_CAMPO=1`, `SOLO_DATI=False`.

**PARTITA INTERA:** `SALTO=3`, `OGNI_CAMPO=2`, **`SOLO_DATI=True`** (niente video radar →
più veloce e leggero). Stima su GPU T4: **~45-60 min** una partita intera.

In [ ]:
SALTO = 2
OGNI_CAMPO = 1
SOLO_DATI = False   # True per PARTITA INTERA (niente video radar, solo report)

import os
video = 'clips_input/clip_input.mp4'
base = os.path.splitext(os.path.basename(video))[0]
csv_pos = f'output/POSIZIONIAUTO_{base}.csv'
novideo = '--no_video' if SOLO_DATI else ''

print('▶ 1/5 Radar/dati + rilevamento giocatori...')
!python vista_tattica/genera_radar_auto.py "{video}" --salto {SALTO} --ogni_campo {OGNI_CAMPO} --imgsz 1280 {novideo}
print('\n▶ 2/5 Heatmap + statistiche...')
!python analisi/stats_giocatori.py "{csv_pos}" --min_rilevazioni 20
print('\n▶ 3/5 Dashboard di confronto...')
!python analisi/confronto_giocatori.py "output/STATISTICHE_{base}.csv"
print('\n▶ 4/5 Metriche scouting (formazione, andamento, atletico)...')
!python analisi/metriche_avanzate.py "{csv_pos}" --min_rilevazioni 20
print('\n▶ 5/5 Report PDF unico...')
!python analisi/report_pdf.py "{base}" --dir output
print('\n✅ FATTO.')

## 5) Anteprima risultati (il PDF completo è nello zip)

In [ ]:
from IPython.display import Image, display
import glob
for pattern, titolo in [('output/FORMAZIONE_*.png', '=== FORMAZIONE / MODULO ==='),
                        ('output/ANDAMENTO_*.png', '=== ANDAMENTO TATTICO ==='),
                        ('output/REPORT_*.png', '=== REPORT SCOUTING ==='),
                        ('output/DASHBOARD_*.png', '=== CONFRONTO GIOCATORI ===')]:
    for f in glob.glob(pattern):
        print(titolo); display(Image(f))

## 6) Scarica il report (zip con PDF + immagini + CSV)

In [ ]:
import shutil
from google.colab import files
shutil.make_archive('report', 'zip', 'output')
files.download('report.zip')